# Follower POGEMA Baseline — Google Colab

Runs the **Follower** baseline (pure-Python, no FollowerLite) across the POGEMA `experiments/` folders, pulling code from the **`tay805/mapf-deadlock`** repo so local changes flow through GitHub.

**Before running:** set `Runtime > Change runtime type > GPU` (T4 is fine).

**Why conda?** The repo is pinned to `torch 1.13.1 / sample-factory 2.1.1 / numpy<=1.23.1`, which need **Python 3.10**. Colab ships 3.11/3.12, so cell 1 installs a 3.10 env via `condacolab` (the kernel will restart — expected; continue with cell 2).

**Speed note:** GPU accelerates the policy net, but A* + env stepping stay CPU-bound and Colab free has ~2 vCPU. The win is offloading your machine, not raw speed.

## 1. Install Python 3.10 env (kernel will restart)

In [ ]:
# Installs a Python 3.10 conda base. The kernel RESTARTS automatically at the end
# of this cell — that is normal. After it restarts, run cell 2 onward.
!pip install -q condacolab
import condacolab
condacolab.install()

## 2. Verify env + GPU (run AFTER the restart)

In [ ]:
import sys
print('Python:', sys.version)   # expect 3.10.x
import condacolab; condacolab.check()
!nvidia-smi -L || echo 'No GPU — set Runtime > Change runtime type > GPU'

## 3. Clone the project repo

Public repo, so no auth needed. Weights are committed (~59MB), so the clone brings them along.

To pull later changes without re-cloning: `!git -C /content/mapf-deadlock pull`

In [ ]:
%cd /content
![ -d mapf-deadlock ] || git clone https://github.com/tay805/mapf-deadlock.git
%cd /content/mapf-deadlock
!ls -lh model/follower/checkpoint_p0/*.pth baseline_eval.py

## 4. Install dependencies (CUDA torch 1.13.1 + pinned requirements)

In [ ]:
# build-essential + pybind11: to JIT-compile the C++ A* planner (cppimport).
!apt-get -qq install -y build-essential >/dev/null
# CUDA torch 1.13.1 (cu117 works on Colab T4 / py3.10).
!pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
# Curated, WHEEL-PREFERRED install. We do NOT use `-r docker/requirements.txt`:
# its conflicting pins make pip backtrack to old versions with no py3.10 wheel,
# which then fail to build from source ("getting requirements to build wheel").
# --prefer-binary keeps pip on wheels. sample-factory pulls its own deps; we add
# the eval's needs on top (and skip tensorboard/protobuf/pytest/wandb — unused).
!pip install --prefer-binary \
  sample-factory==2.1.1 pogema==1.3.0 pogema-toolbox==0.1.0 \
  "numpy>=1.21.5,<=1.23.1" "pandas<=1.4" matplotlib seaborn tabulate \
  pyyaml dask==2024.8.0 distributed==2024.8.0 loguru cppimport pybind11==2.13.1
# Fail loudly HERE if anything is missing, instead of deep inside the eval.
import importlib
missing = []
for m in ['pogema', 'pogema_toolbox', 'sample_factory', 'torch', 'yaml', 'dask', 'loguru', 'cppimport', 'pybind11']:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f'{m}: {e}')
print('MISSING:', missing if missing else 'none — all key imports OK')
import torch; print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())

## 5. Quick validation run (warehouse, 1 seed — a few minutes)

`baseline_eval.py` lives in the repo (Follower-only, no wandb, per-folder JSON, `--seeds=N`). Confirms the stack before the long run.

In [ ]:
!python baseline_eval.py --seeds=1 05-warehouse

## 6. Full 3-seed baseline pass (all folders)

Long-running. Keep the tab active so Colab doesn't disconnect. To split work, pass specific folders, e.g. `!python baseline_eval.py --seeds=3 02-mazes 05-warehouse`.

In [ ]:
!python baseline_eval.py --seeds=3

## 7. Zip + download results

In [ ]:
!cd experiments && zip -qr /content/baseline_results.zip . -i '*.json' '*.csv' '*.svg' '*.png' '*.pdf'
from google.colab import files
files.download('/content/baseline_results.zip')